# Implement Multi-Head Attention from Scratch
### Problem Statement
Multi-Head Attention (MHA) is the bread-and-butter of the Transformer architecture. It enables the model to **jointly attend** to information from different representation subspaces at different positions.

Your goal is to implement the MHA forward pass from scratch in PyTorch, computing exactly what `torch.nn.MultiheadAttention` does: project Q, K and V, split them into heads, run scaled dot-product attention in every head, concatenate the heads and apply the output projection. The projection weights are passed in as arguments, so you can copy them out of an `nn.MultiheadAttention` and compare outputs exactly.

---

### Requirements

1. **Signature**: `multi_head_attention(q, k, v, num_heads, W_q, W_k, W_v, W_o, mask=None)`
   - `q`: `(batch_size, seq_len_q, d_model)`; `k`, `v`: `(batch_size, seq_len_k, d_model)`.
   - `W_q`, `W_k`, `W_v`, `W_o`: `(d_model, d_model)` in `nn.Linear` layout, so a projection is `x @ W.T` (`F.linear(x, W)`).

2. **Linear Projections and Head Split**
   - Project `q`, `k`, `v`, then split `d_model` into `num_heads` heads of `d_head = d_model // num_heads`: shape `(batch_size, num_heads, seq_len, d_head)`.

3. **Scaled Dot-Product Attention per Head**
   - `scores = Q @ Kᵀ / sqrt(d_head)`.
   - Optional boolean `mask`, broadcastable to `(batch_size, num_heads, seq_len_q, seq_len_k)`: **`True` = may attend**, `False` = blocked (`-inf` before the softmax).
   - Softmax over the keys, then weight `V`.

4. **Combine the Heads**
   - Concatenate the heads back to `(batch_size, seq_len_q, d_model)` and apply `W_o`.

5. **Validate Against PyTorch's Reference**
   - Build `nn.MultiheadAttention(d_model, num_heads, bias=False, batch_first=True)` and take `W_q, W_k, W_v = in_proj_weight.chunk(3)` and `W_o = out_proj.weight`.
   - Check `torch.allclose` with and without a causal mask. Careful: `nn.MultiheadAttention`'s boolean `attn_mask` uses the opposite convention (`True` = blocked), so pass it `~mask`.

---

### Constraints

- ✅ Use only PyTorch operations; do not call `nn.MultiheadAttention` or `F.scaled_dot_product_attention` inside your function.
- ✅ Make sure all tensors are reshaped properly when splitting and combining heads.
- ✅ Support optional masking.

---

<details>
  <summary>💡 Hint</summary>

  - Use `.view()` and `.transpose(1, 2)` to shape Q, K, V to `(batch_size, num_heads, seq_len, d_head)`.
  - Softmax should be applied over the **last dimension** (the keys).
  - Use `.transpose(1, 2).contiguous().view()` to merge the heads back into `(batch_size, seq_len_q, d_model)`.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

In [ ]:
def multi_head_attention(q, k, v, num_heads, W_q, W_k, W_v, W_o, mask=None):
    """
    Implements the multi-head attention forward pass.

    Args:
        q (Tensor): Query tensor of shape (batch_size, seq_len_q, d_model)
        k (Tensor): Key tensor of shape (batch_size, seq_len_k, d_model)
        v (Tensor): Value tensor of shape (batch_size, seq_len_k, d_model)
        num_heads (int): Number of attention heads (must divide d_model)
        W_q, W_k, W_v, W_o (Tensor): (d_model, d_model) projection weights in
            nn.Linear layout: a projection is x @ W.T
        mask (Tensor, optional): bool, broadcastable to
            (batch_size, num_heads, seq_len_q, seq_len_k); True = may attend

    Returns:
        Tensor: Multi-head attention output of shape (batch_size, seq_len_q, d_model)
    """
    batch_size, seq_len_q, d_model = q.shape
    seq_len_k = k.shape[1]
    assert d_model % num_heads == 0, "num_heads must divide d_model"
    d_head = d_model // num_heads

    # Project, then split d_model into heads: (batch, num_heads, seq, d_head)
    Q = F.linear(q, W_q).view(batch_size, seq_len_q, num_heads, d_head).transpose(1, 2)
    K = F.linear(k, W_k).view(batch_size, seq_len_k, num_heads, d_head).transpose(1, 2)
    V = F.linear(v, W_v).view(batch_size, seq_len_k, num_heads, d_head).transpose(1, 2)

    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_head)  # (batch, num_heads, seq_q, seq_k)
    if mask is not None:
        scores = scores.masked_fill(~mask.bool(), float("-inf"))
    attn_weights = F.softmax(scores, dim=-1)
    out = attn_weights @ V                                 # (batch, num_heads, seq_q, d_head)

    # Merge the heads back to (batch, seq_q, d_model), then the output projection
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len_q, d_model)
    return F.linear(out, W_o)

In [ ]:
# Testing on data & compare with nn.MultiheadAttention, using the same weights
multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True)
W_q, W_k, W_v = multihead_attn.in_proj_weight.chunk(3)  # each (d_model, d_model)
W_o = multihead_attn.out_proj.weight

with torch.no_grad():
    output_custom = multi_head_attention(q, k, v, num_heads, W_q, W_k, W_v, W_o)
    output, _ = multihead_attn(q, k, v, need_weights=False)
print(output_custom)
print(output)
assert torch.allclose(output_custom, output, atol=1e-6) # Check if they are close enough.

# Causal self-attention. Our mask: True = may attend; nn.MultiheadAttention's: True = blocked.
causal = torch.ones(seq_len, seq_len, dtype=torch.bool).tril()
with torch.no_grad():
    output_custom = multi_head_attention(q, q, q, num_heads, W_q, W_k, W_v, W_o, mask=causal)
    output, _ = multihead_attn(q, q, q, attn_mask=~causal, need_weights=False)
assert torch.allclose(output_custom, output, atol=1e-6)
print("matches nn.MultiheadAttention")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("multi-head-attention", multi_head_attention=multi_head_attention)